This file is as same as the simple agent , but in this we are adding the structured output and the streaming adding the additional features, the rest will remain smae.

In [19]:
# imports and configrations

In [20]:
from langsmith import Client
from pathlib import Path
from dotenv import load_dotenv
import langsmith as ls
import os

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from typing import TypedDict, Literal, Annotated
from typing_extensions import NotRequired

from pydantic import BaseModel, Field
from rich import print
import sys
import time

In [23]:
env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)
openai_api_key = os.getenv("OPENAI_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")


In [24]:
os.environ["LANGSMITH_PROJECT"] = "simple-agent"
os.environ["LANGCHAIN_PROJECT"] = "simple-agent"
os.environ["LANGCHAIN_TRACING_V2"] = "true" 

In [25]:
# business logic

In [27]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=openai_api_key,
    temperature=0
)

In [28]:
class StructuredOutputForToolCall(BaseModel):
    question: str = Field(
        ...,
        description="The original user question or calculation request"
    )
    num: float = Field(
        ...,
        description="Contains the final output as the numerical value from the tool call"
    )

    @property
    def formatted_response(self) -> str:
        return f"For your question '{self.question}' the output is {self.num}"

In [29]:
# defining tools 

In [30]:
@tool
def add(num1:float, num2:float) -> float:
    """This tool is used to add two numbers"""
    return num1 + num2


@tool
def subtract(num1:float, num2:float) -> float:
    """This tool is used to subtract two numbers"""
    return num1 - num2

@tool
def multiply(num1:float, num2:float) -> float:
    """This tool is used to multiply two numbers"""
    return num1 * num2

@tool
def divide(num1:float, num2:float) -> float:
    """This tool is used to divide two numbers"""
    try:
        return num1 / num2
    except ZeroDivisionError:
        print("Error: Cannot divide by zero")
        return None

In [31]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    structured_response: StructuredOutputForToolCall | None = None

In [34]:
# binding the model with the tools and structured output
tools = [add, subtract, multiply, divide]
model_with_tools = model.bind_tools(tools)
structured_model = model.with_structured_output(StructuredOutputForToolCall)

In [35]:
def agent_node(state: State):
    # Since state is a Pydantic model, access attributes via dot notation
    messages = state.messages

    # Check if a tool was just executed (last message is a ToolMessage)
    if isinstance(messages[-1], ToolMessage):
        structured_res = structured_model.invoke(messages)
        content = f"For your question '{structured_res.question}' the output is {structured_res.num}"
        return {
            "messages": [AIMessage(content=content)],
            "structured_response": structured_res
        }

    # General conversation or model deciding whether to invoke a tool
    response = model_with_tools.invoke(messages)
    return {
        "messages": [response],
        "structured_response": None
    }

In [36]:
def should_continue(state: State):
    messages = state.messages
    last_message = messages[-1]

    if getattr(last_message, "tool_calls", None):
        return "tools"

    return "end" 

In [39]:
from langgraph.prebuilt import ToolNode

tools_node = ToolNode(tools)

In [40]:
# Build the graph

In [41]:
# defining the graph
graph = StateGraph(State)

# defining the nodes
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node)

# defining the edges
graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)

# After executing tools, return to the agent
graph.add_edge("tools", "agent")

workflow = graph.compile()

In [45]:
workflow

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [46]:
# making the instance of the state class
state = State()

# defining the configuration for the workflow
config = {
    "configurable": {
        "thread_id": "user_123"
    }
}

In [ ]:
def stream_response(text: str, delay: float = 0.012):
    """Streams text character-by-character to stdout for real-time response delivery."""
    for char in text:
        sys.stdout.write(char)
        sys.stdout.flush()
        time.sleep(delay)
    print()

while True:
    user_query = input("Enter your query: ")

    # Check if the user wants to exit the program
    if user_query.lower() == "exit":
        print("Exiting the program.")
        break

    print(f"\nUser Query: {user_query}")

    human_message = HumanMessage(content=user_query)
    state.messages.append(human_message)

    # Stream the workflow updates in real-time
    for event in workflow.stream(state, config=config, stream_mode="updates"):
        for node_name, node_output in event.items():
            # Keep state messages synchronized with node updates
            if "messages" in node_output:
                state.messages.extend(node_output["messages"])
            if "structured_response" in node_output and node_output["structured_response"] is not None:
                state.structured_response = node_output["structured_response"]

            # Stream progress live for tool calls and all responses
            if node_name == "agent":
                latest_msg = node_output["messages"][-1]
                if getattr(latest_msg, "tool_calls", None):
                    for tc in latest_msg.tool_calls:
                        print(f"⚡ [Streaming] Invoking tool '{tc['name']}' with args: {tc['args']}")
                else:
                    # Stream the response content live in real-time
                    sys.stdout.write("🤖 AI Response: ")
                    sys.stdout.flush()
                    stream_response(latest_msg.content)
            elif node_name == "tools":
                print(f"🔧 [Streaming] Tool execution result: {node_output['messages'][-1].content}")

    # If a tool call occurred, display the structured schema object
    if state.structured_response is not None:
        print(f"📦 Structured Output Schema: {state.structured_response}")
        # Reset for subsequent turns
        state.structured_response = None

    print()

User Query: hello

🤖 AI Response: Hello! How can I assist you today?

User Query: who you are ?

🤖 AI Response: I am ChatGPT, an AI language model created by OpenAI. I'm here to help you with information, answer questions, assist with tasks, and chat about a wide range of topics. How can I assist you today?

User Query: What is your knowledge cutoff date ?

🤖 AI Response: My knowledge is current up until June 2024. If you have any questions or need information based on that timeframe, feel free to ask!

... (6 outputs left out)

⚡ [Streaming] Invoking tool 'subtract' with args: {'num1': 65, 'num2': 2}

🔧 [Streaming] Tool execution result: 63.0

🤖 AI Response: For your question 'What is 35 multiplied by 5 and 65 minus 2 added to it, and then 10 multiplied to the result?' the output is 2380.0

📦 Structured Output Schema: question='What is 35 multiplied by 5 and 65 minus 2 added to it, and then 10 
multiplied to the result?' num=2380.0